# Work-Life Firewall Kaggle Runner

Run this notebook top-to-bottom in Kaggle with GPU enabled.

## Required Kaggle Secrets
- `WANDB_API_KEY`
- `HF_TOKEN` (optional, only needed for model upload)


In [ ]:
import os
import shutil
from pathlib import Path

def find_repo_dir() -> Path:
    # Preferred explicit path first
    candidates = [Path('/kaggle/working/meta_hackathon')]

    # Common Kaggle working directories
    working = Path('/kaggle/working')
    if working.exists():
        candidates.append(working)
        candidates.extend([p for p in working.iterdir() if p.is_dir()])

    # Attached Kaggle datasets are often mounted under /kaggle/input
    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.append(input_root)
        for ds in input_root.iterdir():
            if ds.is_dir():
                candidates.append(ds)
                candidates.extend([p for p in ds.iterdir() if p.is_dir()])

    for c in candidates:
        if (c / 'openenv.yaml').exists() and (c / 'training').exists():
            return c

    raise FileNotFoundError(
        'Could not find project root with openenv.yaml + training/. '
        'If your repo is not attached yet, attach/upload it first.'
    )

REPO_DIR = find_repo_dir()

# /kaggle/input is read-only; copy to /kaggle/working for training outputs.
if str(REPO_DIR).startswith('/kaggle/input'):
    target = Path('/kaggle/working/meta_hackathon')
    if not target.exists():
        shutil.copytree(REPO_DIR, target)
    REPO_DIR = target

os.chdir(REPO_DIR)
print('CWD:', Path.cwd())
print('Found project root:', REPO_DIR)
!python --version
!nvidia-smi

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# WandB token (required for tracking)
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'meta_hackathon'

# HF token is optional; needed only if you push model from Kaggle
try:
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['HUGGINGFACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF token loaded from Kaggle Secrets')
except Exception:
    print('HF token not set (this is okay if you are not uploading model now)')

print('WandB key loaded:', bool(os.environ.get('WANDB_API_KEY')))

In [ ]:
# Fast stable run for Kaggle T4
!python training/train.py --mode real --speed-preset fast --size-preset small --steps 50 --run-name meta-kaggle-final --wandb-project meta_hackathon

In [ ]:
import json
from pathlib import Path

metrics_path = Path('evaluation/results/training_metrics.json')
metrics = json.loads(metrics_path.read_text())
print('Metrics file:', metrics_path)
print('Mode:', metrics.get('mode'))
print('Model:', metrics.get('model_id'))
print('Train runtime (s):', metrics.get('train_runtime_seconds'))
print('WandB URL:', metrics.get('wandb_run_url'))
print('Loss points:', len(metrics.get('loss_curve', [])))
print('Reward points:', len(metrics.get('reward_curve', [])))

In [ ]:
!python evaluation/evaluate.py --episodes 50
!python evaluation/plot_results.py
!ls -lah evaluation/results

In [ ]:
# Package submission artifacts for download
!zip -r submission_artifacts.zip README.md openenv.yaml environment training evaluation/results
!ls -lah submission_artifacts.zip

In [ ]:
# Optional: upload model to HF Model Hub
# Uncomment and edit repo id when needed
# !python training/push_to_hf.py --repo-id YUS200619/meta_hackathon-qwen-model --folder checkpoints/final